In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

In [2]:
#1. Load data

In [3]:

file_path = r"C:\Users\kone\OneDrive - Seriti Business Solutions, Pty (Ltd)\Documents\STADIO\Capstone_Project_KM\Preprocessing\bank-full.csv"

df = pd.read_csv(file_path, sep=";")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
#inspect the dataset

In [5]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

Columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Data types:


age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object


Missing values:


age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [6]:
X = df.drop(columns=["y"])
y = df["y"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print(y.value_counts(normalize=True) * 100)

Predictor shape: (45211, 16)
Target shape: (45211,)

Target distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target percentages:
y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64


In [7]:
#train_test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training set: (36168, 16)
Testing set: (9043, 16)

Training target distribution:
y
no     88.301814
yes    11.698186
Name: proportion, dtype: float64

Testing target distribution:
y
no     88.300343
yes    11.699657
Name: proportion, dtype: float64


In [8]:
#creating the features
def engineer_features(X):
    
    X_engineered = X.copy()

    # 1. Age Group
    X_engineered["age_group"] = pd.cut(
        X_engineered["age"],
        bins=[0, 25, 35, 45, 55, 65, np.inf],
        labels=[
            "18-25",
            "26-35",
            "36-45",
            "46-55",
            "56-65",
            "66+"
        ]
    )

    # 2. Has Balance
    X_engineered["has_balance"] = (
        X_engineered["balance"] > 0
    ).astype(int)

    # 3. Balance-to-Age Ratio
    X_engineered["balance_to_age"] = (
        X_engineered["balance"] /
        (X_engineered["age"] + 1)
    )

    # 4. Has Previous Contact
    X_engineered["has_previous_contact"] = (
        X_engineered["pdays"] != -1
    ).astype(int)

    # 5. Previous Positive Outcome
    X_engineered["previous_positive_outcome"] = (
        X_engineered["poutcome"] == "success"
    ).astype(int)

    # 6. Contact Intensity
    X_engineered["contact_intensity"] = pd.cut(
        X_engineered["campaign"],
        bins=[0, 1, 3, np.inf],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    )

    return X_engineered

In [9]:
#apply feature engineering
X_train_engineered = engineer_features(X_train)

X_test_engineered = engineer_features(X_test)

print("Original training features:", X_train.shape[1])
print("Engineered training features:", X_train_engineered.shape[1])

print("Original testing features:", X_test.shape[1])
print("Engineered testing features:", X_test_engineered.shape[1])

Original training features: 16
Engineered training features: 22
Original testing features: 16
Engineered testing features: 22


In [10]:
#inspect new features
engineered_features = [
    "age_group",
    "has_balance",
    "balance_to_age",
    "has_previous_contact",
    "previous_positive_outcome",
    "contact_intensity"
]

X_train_engineered[engineered_features].head(10)

,age_group,has_balance,balance_to_age,has_previous_contact,previous_positive_outcome,contact_intensity
24001,36-45,1,23.270270,0,0,Medium
43409,18-25,1,165.040000,1,0,High
20669,36-45,1,5.422222,0,0,High
18810,46-55,0,0.000000,0,0,High
23130,36-45,1,6.589744,0,0,High
15058,46-55,1,30.877551,0,0,Low
15908,46-55,1,84.607843,0,0,High
30424,46-55,0,-16.595745,0,0,Low
9998,46-55,1,10.085106,0,0,Medium
14935,36-45,1,5.391304,0,0,Medium


In [11]:
#checking engineered features
print("Age groups:")
display(X_train_engineered["age_group"].value_counts())

print("\nCustomers with balance:")
display(X_train_engineered["has_balance"].value_counts())

print("\nPrevious contact:")
display(X_train_engineered["has_previous_contact"].value_counts())

print("\nPrevious positive outcome:")
display(X_train_engineered["previous_positive_outcome"].value_counts())

print("\nContact intensity:")
display(X_train_engineered["contact_intensity"].value_counts())

Age groups:


age_group
26-35    12511
36-45    11098
46-55     7589
56-65     3282
18-25     1079
66+        609
Name: count, dtype: int64


Customers with balance:


has_balance
1    30358
0     5810
Name: count, dtype: int64


Previous contact:


has_previous_contact
0    29584
1     6584
Name: count, dtype: int64


Previous positive outcome:


previous_positive_outcome
0    34963
1     1205
Name: count, dtype: int64


Contact intensity:


contact_intensity
Medium    14435
Low       14026
High       7707
Name: count, dtype: int64

In [12]:
#check for missing values created by engineering
print("Missing values after feature engineering:")

display(
    X_train_engineered.isnull().sum()
    .sort_values(ascending=False)
)

Missing values after feature engineering:


age                          0
job                          0
marital                      0
education                    0
default                      0
balance                      0
housing                      0
loan                         0
contact                      0
day                          0
month                        0
duration                     0
campaign                     0
pdays                        0
previous                     0
poutcome                     0
age_group                    0
has_balance                  0
balance_to_age               0
has_previous_contact         0
previous_positive_outcome    0
contact_intensity            0
dtype: int64

In [15]:
#save engineered datasets
X_train_engineered.to_csv(
    r"C:\Users\kone\OneDrive - Seriti Business Solutions, Pty (Ltd)\Documents\STADIO\Capstone_Project_KM\Preprocessing\X_train_engineered.csv",
    index=False
)

X_test_engineered.to_csv(
    r"C:\Users\kone\OneDrive - Seriti Business Solutions, Pty (Ltd)\Documents\STADIO\Capstone_Project_KM\Preprocessing\X_test_engineered.csv",
    index=False
)

y_train.to_csv(
    r"C:\Users\kone\OneDrive - Seriti Business Solutions, Pty (Ltd)\Documents\STADIO\Capstone_Project_KM\Preprocessing\y_train.csv",
    index=False
)

y_test.to_csv(
    r"C:\Users\kone\OneDrive - Seriti Business Solutions, Pty (Ltd)\Documents\STADIO\Capstone_Project_KM\Preprocessing\y_test.csv",
    index=False
)

print("Feature-engineered datasets saved successfully.")

Feature-engineered datasets saved successfully.
